In [1]:
import jax
from jax import lax
from jax import numpy as jnp
from jax.tree_util import Partial

import polars as pl
import re
from matplotlib import pyplot as plt

from gm_scenario import *
from ASK14 import ASK14

In [33]:
gmc = pl.read_csv('BSSA14_coeffs.csv')

gmc[-2, 'T'] = -1.
gmc[-1, 'T'] = -2.
gmc_col = gmc.columns
gmc = gmc.cast(pl.Float64).to_jax().T

print(gmc_col)
print(gmc_col[8:12])
T = gmc[0]
empty = jnp.zeros_like(T, dtype = float)
e = gmc[1:8]
Mh = gmc[8]
c = gmc[9:12]
c = jnp.insert(c, 0, empty, 0)
M_ref, R_ref, h, Dc3CaTw, Dc3CnTr, Dc3ItJp = gmc[12:18]
c, Vc, V_ref, = gmc[18:21]
f = gmc[21:27]
f = jnp.insert(f, 0, empty, axis = 0)
f = jnp.insert(f, 2, empty, axis = 0)
R1, R2 = gmc[27:29]
dPhi_R, dPhi_V = gmc[29:31]
v1, v2, phi1, phi2, tau1, tau2 = gmc[31:]

['T', 'e0', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'Mh', 'c1', 'c2', 'c3', 'Mref', 'Rref', 'h', 'Dc3CaTw', 'Dc3CnTr', 'Dc3ItJp', 'c', 'Vc', 'Vref', 'f1', 'f3', 'f4', 'f5', 'f6', 'f7', 'R1', 'R2', 'dPhiR', 'dPhiV', 'v1', 'v2', 'phi1', 'phi2', 'tau1', 'tau2']
['Mh', 'c1', 'c2', 'c3']


In [ ]:
# Source model (eqn. 2)
def F_event(Mw, SOF):
    # Select last term
    e_last = lax.select(Mw <= Mh, 
                        e[4] * (Mw - Mh) + e[5] * (Mw - Mh) ** 2,
                        e[6] * (Mw - Mh))
    
    # Build SOF condition ([-1, 0, 1, 2] -> [0, 1, 2, 3])
    cond_SOF = SOF + 1
    e_SOF = lax.select_n(cond_SOF, e[:4])
    return e_SOF + e_last

# Path model (eqns. 3, 4)
def F_path(Mw, R_jb, region):
    R = (R_jb ** 2 + h ** 2) ** (1 / 2)
    cond = 0
    Dc3 = lax.select_n(cond, empty, Dc3CaTw, Dc3CnTr, Dc3ItJp)
    return jnp.log(R / R_ref) * (c[1] + c[2] * (Mw - M_ref)) + (c[3] + Dc3) * (R - R_ref)

# Site model (eqn. 5 - 12): 
def F_site(Mw, R_jb, vs30, z1p0, region, PGA_r):
    vs30_capped = jnp.clip(vs30, max = Vc)
    lnF_lin = c * jnp.log(vs30_capped / V_ref)

    lnF_nl = f[1] + f[2] * jnp.log(PGA_r / f[3] + 1)



"""
true_ask14:
- Mandatory:
    - dip (float)
    - dist_jb (float)
    - dist_rup (float)
    - dist_x (float)
    - mag (float)
    - mechanism (str, "SS", "NS", "RS")
    - v_s30
- Optional:
    - depth_1_0 (float)
    - depth_2_5 (float)
    - depth_tor (float)
    - depth_bor (float)
    - dist_y0 (float)
    - on_hanging_wall (bool, None = false)
    - region (str, 'global', 'cali', 'china', 'italy', 'japan', 'taiwan')
    - vs_source (str, 'measured', 'inferred')
    - width (float, none = model average)
"""